# A trapped-ion qubit: Rabi flopping, Ramsey fringes, spin echoes and decay

**Professor Nano Labs · Quantum computing**

**Try it live in your browser:** [Trapped-Ion Qubit lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/trapped-ion-qubit/)
**Watch first:** [How exactly does trapped ions perform quantum computing?](https://www.youtube.com/watch?v=5qpXdGSLWig)
**Watch later:** [How to trap ions for quantum computing?](https://www.youtube.com/watch?v=8p6P-QTQM2M) · [Quick introduction to the density matrix in quantum mechanics](https://www.youtube.com/watch?v=ma136N8d3Xc) · [Mapping the qubit state onto the Bloch Sphere](https://www.youtube.com/watch?v=lqWSziZJsLs)

A trapped ion stores a qubit in two long-lived levels: $|0\rangle$, the ground state, and $|1\rangle$, an excited state. A laser close to the transition frequency $\omega_0$ drives the ion between them. In the frame rotating with the laser, after the rotating-wave approximation,

$$H=\frac{\hbar}{2}\,\vec W\cdot\vec\sigma,\qquad \vec W=(\Omega\cos\phi,\ \Omega\sin\phi,\ \Delta),\qquad \Delta=\omega_{\rm laser}-\omega_0 ,$$

so the Bloch vector precesses about $\vec W$. Adding energy loss ($T_1$) and dephasing ($T_2$) gives the **Bloch equations**

$$\frac{d\vec r}{dt}=\vec W\times\vec r-\left(\frac{r_x}{T_2},\ \frac{r_y}{T_2},\ \frac{r_z-1}{T_1}\right),$$

which relax toward the ground state $\vec r=(0,0,1)$. This notebook solves them exactly, checks them against the Lindblad master equation for the density matrix, and reproduces the lab's three experiments. It needs only NumPy and Matplotlib. **Run all cells from the top.**

Units: time in µs and frequencies in kHz, so an angular frequency is $2\pi f/1000$ rad/µs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

def kHz(f):
    # frequency in kHz -> angular frequency in rad/us
    return 2 * np.pi * f / 1000

X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]])
Z = np.diag([1.0, -1.0]).astype(complex)
I2 = np.eye(2, dtype=complex)

def expm(A):
    # matrix exponential by scaling and squaring with a Taylor series
    norm = np.abs(A).sum(axis=1).max()
    s = max(0, int(np.ceil(np.log2(norm / 0.5)))) if norm > 0.5 else 0
    A = A / 2**s
    E, term = np.eye(len(A), dtype=A.dtype), np.eye(len(A), dtype=A.dtype)
    for k in range(1, 25):
        term = term @ A / k
        E = E + term
    for _ in range(s):
        E = E @ E
    return E

## 1. Rabi flopping

With no decay, the evolution over a time $t$ is a rotation by $|\vec W|t$ about $\vec W$: $U=e^{-iHt/\hbar}=R_{\hat W}(|\vec W|t)$. Starting in $|0\rangle$, the excitation probability is the **Rabi formula**

$$P_1(t)=\frac{\Omega^2}{\Omega^2+\Delta^2}\,\sin^2\!\left(\frac{\sqrt{\Omega^2+\Delta^2}\,t}{2}\right).$$

On resonance the ion is certainly excited after a **π pulse**, $t_\pi=\pi/\Omega$. Off resonance it never is.

In [ ]:
def U_drive(omega, delta, phase, t):
    # exp(-i H t / hbar) for H = (hbar/2) W.sigma
    W = np.array([omega * np.cos(phase), omega * np.sin(phase), delta])
    w = np.linalg.norm(W)
    if w == 0:
        return I2
    n = W / w
    return np.cos(w * t / 2) * I2 - 1j * np.sin(w * t / 2) * (n[0] * X + n[1] * Y + n[2] * Z)

def rabi_formula(omega, delta, t):
    we = np.hypot(omega, delta)
    return omega**2 / we**2 * np.sin(we * t / 2)**2

omega = kHz(100)                 # Omega / 2pi = 100 kHz
t_pi = np.pi / omega
print(f'pi pulse at 100 kHz: {t_pi:.3f} us')

ket0 = np.array([1, 0], dtype=complex)
t = np.linspace(0, 50, 1001)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
worst = 0
for d_kHz, col in [(0, '#a3326c'), (50, '#1f6fb4'), (100, '#e07b1f')]:
    P1 = np.array([abs((U_drive(omega, kHz(d_kHz), 0, x) @ ket0)[1])**2 for x in t])
    worst = max(worst, np.abs(P1 - rabi_formula(omega, kHz(d_kHz), t)).max())
    a1.plot(t, P1, color=col, lw=2, label=f'Δ/2π = {d_kHz} kHz')
print('max |propagated - Rabi formula|:', f'{worst:.1e}')
a1.set_xlabel('Time t (µs)'); a1.set_ylabel('P(1)'); a1.set_title('Rabi flopping at Ω/2π = 100 kHz')
a1.set_ylim(0, 1.3); a1.legend(loc='upper center', ncol=3, fontsize=8); a1.grid(alpha=0.3)

# spectroscopy: P(1) after a pi pulse against detuning
f = np.linspace(-400, 400, 801)
for om_kHz, col in [(100, '#a3326c'), (50, '#1f6fb4')]:
    om = kHz(om_kHz)
    a2.plot(f, rabi_formula(om, kHz(f), np.pi / om), color=col, lw=2, label=f'Ω/2π = {om_kHz} kHz')
    for z in (-1, 1):
        a2.axvline(z * np.sqrt(3) * om_kHz, color=col, ls=':', lw=1)
a2.set_xlabel('Detuning Δ/2π (kHz)'); a2.set_ylabel('P(1) after a π pulse'); a2.set_title('π-pulse spectrum: zeros at ±√3 Ω')
a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Decay: the Bloch equations and the master equation

Written for $[\vec r;1]$, the Bloch equations are linear with a constant 4 × 4 generator $M$ during each pulse or wait, so $[\vec r(t);1]=e^{Mt}[\vec r(0);1]$ exactly.

As an independent check, the same physics follows from the **Lindblad master equation** for the density matrix,

$$\dot\rho=-\tfrac{i}{\hbar}[H,\rho]+\sum_k\left(L_k\rho L_k^\dagger-\tfrac12\{L_k^\dagger L_k,\rho\}\right),\qquad L_1=\sqrt{1/T_1}\,|0\rangle\langle1|,\quad L_2=\sqrt{\tfrac12\left(\tfrac{1}{T_2}-\tfrac{1}{2T_1}\right)}\,\sigma_z .$$

$L_1$ is energy loss from $|1\rangle$ to $|0\rangle$ and $L_2$ is pure dephasing. The cell compares the two on a detuned, damped Rabi oscillation.

In [ ]:
def rates(T1, T2):
    g1 = 0.0 if np.isinf(T1) else 1 / T1
    g2 = max(0.0 if np.isinf(T2) else 1 / T2, g1 / 2)     # T2 can never exceed 2 T1
    return g1, g2

def generator(W, g1, g2):
    wx, wy, wz = W
    return np.array([[-g2, -wz,  wy, 0],
                     [ wz, -g2, -wx, 0],
                     [-wy,  wx, -g1, g1],
                     [  0,   0,   0, 0]])

def bloch_evolve(r, W, T1, T2, t):
    g1, g2 = rates(T1, T2)
    return (expm(generator(W, g1, g2) * t) @ np.append(r, 1))[:3]

def liouvillian(W, T1, T2):
    g1, g2 = rates(T1, T2)
    H = 0.5 * (W[0] * X + W[1] * Y + W[2] * Z)
    Ls = [np.sqrt(g1) * np.array([[0, 1], [0, 0]], dtype=complex), np.sqrt((g2 - g1 / 2) / 2) * Z]
    L = -1j * (np.kron(I2, H) - np.kron(H.T, I2))          # column-stacked vec(rho)
    for J in Ls:
        JdJ = J.conj().T @ J
        L = L + np.kron(J.conj(), J) - 0.5 * np.kron(I2, JdJ) - 0.5 * np.kron(JdJ.T, I2)
    return L

def lindblad_evolve(rho, W, T1, T2, t):
    v = expm(liouvillian(W, T1, T2) * t) @ rho.reshape(-1, order='F')
    return v.reshape(2, 2, order='F')

bloch_of = lambda rho: np.real([np.trace(rho @ s) for s in (X, Y, Z)])
rho_of = lambda r: 0.5 * (I2 + r[0] * X + r[1] * Y + r[2] * Z)

W = [kHz(90) * np.cos(0.7), kHz(90) * np.sin(0.7), kHz(-35)]
T1, T2 = 180, 120
r0 = np.array([0, 0, 1.0])
ts = np.linspace(0, 120, 241)
rB = np.array([bloch_evolve(r0, W, T1, T2, x) for x in ts])
rL = np.array([bloch_of(lindblad_evolve(rho_of(r0), W, T1, T2, x)) for x in ts])
print('max |Bloch equations - Lindblad| over the run:', f'{np.abs(rB - rL).max():.1e}')

plt.figure(figsize=(7, 3.6))
plt.plot(ts, (1 - rB[:, 2]) / 2, color='#a3326c', lw=2, label='P(1), T₁ = 180 µs, T₂ = 120 µs')
plt.plot(ts, rabi_formula(kHz(90), kHz(-35), ts), '--', color='0.6', lw=1.2, label='no decay')
plt.xlabel('Time t (µs)'); plt.ylabel('P(1)'); plt.legend(loc='upper right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Ramsey fringes, field noise and the spin echo

**Ramsey:** a π/2 pulse, a wait $\tau$ with the laser off, and another π/2 pulse. During the wait the arrow precesses around the equator by $\Delta\tau$, and the second pulse turns that phase into a population.

**Field noise:** if the detuning drifts slowly, it is constant during one shot but different from shot to shot. Averaging over a Gaussian spread $\sigma$ washes the fringes out as $e^{-\sigma^2\tau^2/2}$.

**Spin echo:** a π pulse halfway through the wait flips the arrows over, so whatever each shot gained in the first half it loses in the second. The echo undoes slow noise but not the fast, random dephasing described by $T_2$.

The lab averages over 41 detunings evenly spaced over ±4σ; so does this cell.

In [ ]:
def run(segments, omega, delta, T1, T2, r=(0, 0, 1.0)):
    # segments: list of (kind, duration, phase); kind 'p' = laser pulse, 'w' = wait
    r = np.array(r, float)
    for kind, dur, ph in segments:
        W = [omega * np.cos(ph), omega * np.sin(ph), delta] if kind == 'p' else [0, 0, delta]
        r = bloch_evolve(r, W, T1, T2, dur)
    return r

def ensemble_run(segments, omega, delta, sigma, T1, T2, K=41):
    if sigma == 0:
        return run(segments, omega, delta, T1, T2)
    x = np.linspace(-4, 4, K); w = np.exp(-x**2 / 2); w /= w.sum()
    return sum(wk * run(segments, omega, delta + sigma * xk, T1, T2) for xk, wk in zip(x, w))

omega = kHz(100); tp = np.pi / omega
ramsey = lambda tau: [('p', tp / 2, 0), ('w', tau, 0), ('p', tp / 2, 0)]
echo = lambda tau: [('p', tp / 2, 0), ('w', tau / 2, 0), ('p', tp, 0), ('w', tau / 2, 0), ('p', tp / 2, np.pi)]
P1 = lambda r: (1 - r[2]) / 2

taus = np.linspace(0, 300, 121)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
for sig, T2, ls in [(3, np.inf, '-'), (3, 100, '--')]:
    lab = f'σ/2π = {sig} kHz' + ('' if np.isinf(T2) else f', T₂ = {T2} µs')
    a1.plot(taus, [P1(ensemble_run(ramsey(x), omega, kHz(10), kHz(sig), np.inf, T2)) for x in taus], ls, color='#a3326c', lw=2, label='Ramsey, ' + lab)
    a1.plot(taus, [P1(ensemble_run(echo(x), omega, kHz(10), kHz(sig), np.inf, T2)) for x in taus], ls, color='#1f6fb4', lw=2, label='echo, ' + lab)
a1.axhline(0.5, color='0.6', ls=':')
a1.set_xlabel('Wait τ (µs)'); a1.set_ylabel('Final P(1)'); a1.set_title('Δ/2π = 10 kHz'); a1.legend(fontsize=8); a1.grid(alpha=0.3)

# Ramsey fringes against detuning, compared with the single pi-pulse line
f = np.linspace(-120, 120, 961)
a2.plot(f, [P1(run(ramsey(50), omega, kHz(x), np.inf, np.inf)) for x in f], color='#a3326c', lw=1.6, label='Ramsey, τ = 50 µs')
a2.plot(f, rabi_formula(omega, kHz(f), tp), color='0.5', lw=1.6, ls='--', label='single π pulse')
a2.set_xlabel('Detuning Δ/2π (kHz)'); a2.set_ylabel('Final P(1)'); a2.set_title('Fringes 1/τ = 20 kHz apart'); a2.legend(fontsize=8); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# compare the averaged Ramsey contrast with exp(-sigma^2 tau^2 / 2) for very short pulses
fast = kHz(20000)
for tau in (50, 100):
    r_end = ensemble_run([('p', np.pi / fast / 2, 0), ('w', tau, 0)], fast, 0, kHz(3), np.inf, np.inf)
    print(f'tau = {tau} us: contrast {np.hypot(r_end[0], r_end[1]):.4f}   exp(-sigma^2 tau^2/2) = {np.exp(-(kHz(3) * tau)**2 / 2):.4f}')

## 4. T₁, T₂ and the density matrix

Leave the ion alone after a pulse. The population decays as $P_1(t)=P_1(0)e^{-t/T_1}$ and the coherence as $|\rho_{01}(t)|=|\rho_{01}(0)|e^{-t/T_2}$. Inside the sphere the state is mixed: its purity $\mathrm{Tr}\,\rho^2=(1+|\vec r|^2)/2$ is below 1.

**Why $T_2\le 2T_1$.** Energy loss by itself destroys coherence at the rate $1/2T_1$. A "$T_2$" longer than $2T_1$ would need negative pure dephasing, and the last cell shows what goes wrong: a state just off the ground state would get a Bloch vector longer than 1, which no density matrix can have.

In [ ]:
T1 = 300
t = np.linspace(0, 1500, 301)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, first) in zip(axes, [('π pulse', [('p', tp, 0)]), ('π/2 pulse', [('p', tp / 2, 0)])]):
    r1 = run(first, omega, 0, T1, np.inf)                        # T2 = inf -> capped at 2 T1
    rs = np.array([bloch_evolve(r1, [0, 0, 0], T1, np.inf, x) for x in t])
    ax.plot(t, (1 - rs[:, 2]) / 2, color='#a3326c', lw=2, label='P(1)')
    ax.plot(t, np.hypot(rs[:, 0], rs[:, 1]), color='#e07b1f', lw=2, label='coherence |r⊥|')
    ax.plot(t, (1 + (rs**2).sum(axis=1)) / 2, color='#1f6fb4', lw=2, label='purity Tr ρ²')
    ax.set_title(f'After a {name}: T₁ = 300 µs, T₂ = 2T₁'); ax.set_xlabel('Time after the pulse (µs)')
    ax.set_ylim(0, 1.05); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

r1 = run([('p', tp, 0)], omega, 0, T1, np.inf)
r300 = bloch_evolve(r1, [0, 0, 0], T1, np.inf, 300)
print('P(1) 300 us after a pi pulse:', round(P1(r300), 4), ' P(1)(0) e^-1 =', round(P1(r1) * np.exp(-1), 4))
print('density matrix then:\n', rho_of(r300))
r_half = run([('p', tp / 2, 0)], omega, 0, T1, np.inf)
r600 = bloch_evolve(r_half, [0, 0, 0], T1, np.inf, 600)
print('coherence 600 us after a pi/2 pulse:', round(np.hypot(*r600[:2]), 4), ' e^-1 x start:', round(np.hypot(*r_half[:2]) * np.exp(-1), 4))

# T2 > 2 T1 breaks physics: start slightly off the ground state and use T2 = 1000 us with T1 = 300 us
def bloch_unchecked(r, T1, T2, t):
    M = np.array([[-1 / T2, 0, 0, 0], [0, -1 / T2, 0, 0], [0, 0, -1 / T1, 1 / T1], [0, 0, 0, 0]])
    return (expm(M * t) @ np.append(r, 1))[:3]
th = 0.3
r_start = np.array([np.sin(th), 0, np.cos(th)])
lengths = [np.linalg.norm(bloch_unchecked(r_start, 300, 1000, x)) for x in np.linspace(0, 1500, 151)]
print(f'\nwith T2 = 1000 us > 2 T1: largest |r| = {max(lengths):.4f} (> 1 is impossible);',
      'smallest eigenvalue of rho:', round(min(np.linalg.eigvalsh(rho_of(bloch_unchecked(r_start, 300, 1000, 300)))), 4))

## 5. Checks

The numbers quoted in the lab's "Predict first" answers, recomputed here.

In [ ]:
omega = kHz(100); tp = np.pi / omega
# Q1: pi pulse 5 us; a 2 pi pulse returns to |0>
assert np.isclose(tp, 5) and np.isclose(rabi_formula(omega, 0, 2 * tp), 0)
# Q2: Delta = Omega -> peak 1/2, flopping at sqrt(2) x 100 kHz
assert np.isclose(omega**2 / (2 * omega**2), 0.5)
print('Q2 flopping rate:', round(np.sqrt(2) * 100, 1), 'kHz')
# Q3: first zeros of the pi-pulse spectrum at +-sqrt(3) Omega
assert abs(rabi_formula(omega, np.sqrt(3) * omega, tp)) < 1e-12
print('Q3 zeros at', round(np.sqrt(3) * 100, 1), 'and', round(np.sqrt(3) * 50, 1), 'kHz')
# Q4: Ramsey, tau = 50 us, Delta/2pi = 10 kHz: Delta tau = pi
print('Q4 final P(1):', round(P1(run(ramsey(50), omega, kHz(10), np.inf, np.inf)), 4))
# Q5: contrast with field noise; the echo stays near 1
print('Q5 exp(-sigma^2 tau^2 / 2) at 50 and 100 us:', round(np.exp(-(kHz(3) * 50)**2 / 2), 3), round(np.exp(-(kHz(3) * 100)**2 / 2), 3))
e5 = [P1(ensemble_run(echo(x), omega, kHz(10), kHz(3), np.inf, np.inf)) for x in (0, 100, 200, 300)]
print('Q5 echo final P(1) at 0, 100, 200, 300 us:', np.round(e5, 4))
# Q6: echo with T2 = 100 us at tau = 100 us
print('Q6 echo final P(1):', round(P1(ensemble_run(echo(100), omega, kHz(10), kHz(3), np.inf, 100)), 4))
# Q7, Q8: T1 = 300 us
assert abs(P1(r300) - 0.37) < 0.01
assert np.isclose(1 / rates(300, np.inf)[1], 600)
print('All checks passed.')

## Things to try

Each idea matches a "Predict first" question in the [browser lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/trapped-ion-qubit/).

1. **Pulse area.** Plot $P_1$ against pulse length on resonance and mark $t_\pi$, $2t_\pi$, $3t_\pi$. Which pulse lengths act as an X gate?
2. **Phase is the axis.** Run a π/2 pulse with `phase = 0` and with `phase = np.pi/2`. Which Bloch-sphere rotation does each one make?
3. **Clock precision.** Compare the width of the Ramsey fringe with the width of the π-pulse line for τ = 50, 100 and 200 µs.
4. **Echo limits.** Hold σ fixed and lower $T_2$. At what wait does the echo signal fall to 0.6?
5. **Unphysical T₂.** Repeat the last cell with `T2 = 600` (exactly $2T_1$). Does $|\vec r|$ still exceed 1?